# Midterm Part A – thin Colab launcher
Clones the fork, installs `midterm_a/requirements.txt` and runs `train.py`. All logic lives in the repo.

Runtime → Change runtime type → **T4 GPU**. Checkpoints, results and the dataset cache live on Google Drive, so after a disconnect re-run the setup cells and the same command: runs resume from `ckpt/last.ckpt` and finished work is skipped.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
FORK_URL = "https://github.com/rafihi/representation-learning-course.git"
WORK = "/content/drive/MyDrive/ssl_midterm_a"   # runs/, results/, data_cache/ on Drive
!mkdir -p {WORK}/runs {WORK}/results {WORK}/data_cache
!rm -rf /content/repo && git clone --filter=blob:none --sparse {FORK_URL} /content/repo
%cd /content/repo
!git sparse-checkout set midterm_a
%cd /content/repo/midterm_a
!ln -s {WORK}/runs runs && ln -s {WORK}/results results && ln -s {WORK}/data_cache data_cache
!pip install -q -r requirements.txt
!pip freeze > {WORK}/requirements.lock.txt   # exact environment (torch included): download and commit as midterm_a/requirements.lock.txt
!git log -1 --format='commit %H'
import torch; print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## 0. Sanity check (synthetic data, ~1 min)

In [ ]:
!SSL_FAKE_DATA=1 SSL_TINY_BACKBONE=1 python -m pytest -q tests

## A2 – tuning (proxy) + reconstruction-vs-k-NN figure
Prints the best trial at the end (already copied into `configs/a2_dae_final.yaml`).

In [ ]:
!python train.py --config configs/a1_tune_dae.yaml --set trainer.progress=false

## A1 / A3 – final DAE and the vanilla-AE baseline (full setting)

In [ ]:
!python train.py --config configs/a2_dae_final.yaml

In [ ]:
!python train.py --config configs/a3_vanilla_ae_final.yaml

## A3 – frozen protocol (k-NN + linear probe) vs. baselines

In [ ]:
!python train.py --config configs/a4_eval.yaml

## A3 / A4 – fine-tuning 1/10/100 % + label-efficiency figure

In [ ]:
!python train.py --config configs/a5_finetune.yaml

## A4 – corruption curves (+ bonus `patch_shuffle`)

In [ ]:
!python train.py --config configs/a6_robustness.yaml

## A5 – per-class accuracy, t-SNE, retrieval grids

In [ ]:
!python train.py --config configs/a7_interpret.yaml

## Final numbers on the TEST split (once, after every decision is made on validation)

In [ ]:
!bash scripts/run_final_test.sh

## Logs

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs